In [ ]:
# !pip install -q unsloth==2025.4.7 datasets==3.5.1

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
from unsloth import FastLanguageModel


MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "up_proj",
        "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state = 42,
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

dataset_name = "5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca"
SYS_INSTRUCT = "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."
raw_dataset = load_dataset(dataset_name, split="train")

def convert_to_chat_format(conversations):
    messages = [{"role": "system", "content": SYS_INSTRUCT}]
    for msg in conversations:
        role = "user" if msg["from"] == "human" else "assistant"
        messages.append({"role": role, "content": msg["value"]})
    return messages

def format_prompt(example):
    messages = convert_to_chat_format(example["conversations"])
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

dataset = raw_dataset.map(format_prompt, remove_columns=raw_dataset.column_names)
dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    save_total_limit=4,
    logging_steps=20,
    output_dir="./checkpoint/llama3-1b-multi-conversation",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    remove_unused_columns=True,
    max_steps=400,
    bf16=True,
)

trainer=SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# load model from ckpt dir then save to huggingface hub
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    "./checkpoint/llama3-1b-multi-conversation/checkpoint-400",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

In [ ]:
type(model)

In [ ]:
model.push_to_hub_merged(
    "thuanan/Llama-3.2-1B-Instruct-Chat-sft",
    commit_message="Merge weights to push to hub",
)

In [ ]:
type(tokenizer)

In [ ]:
tokenizer.push_to_hub(
    "thuanan/Llama-3.2-1B-Instruct-Chat-sft",
    commit_message="Push tokenizer to hub",
)

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=128,
    temperature=1.0,
    do_sample=False,
    num_return_sequences=1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.3
)


prompt = [
    {
        "role": "system",
        "content": "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."
    },
    {
        "role": "user",
        "content": """Hãy chỉnh sửa câu này để ngắn gọn hơn mà không mất đi ý nghĩa: "Trận đấu là một thất bại nặng nề mặc dù thực tế là cả đội đã tập luyện trong nhiều tuần."""
    },
    {
        "role": "assistant",
        "content": """Nhiều tuần huấn luyện của đội đã dẫn đến một thất bại nặng nề."""
    },
    {
        "role": "user",
        "content": """Bạn có thể đề xuất một số chiến lược mà nhóm có thể sử dụng để cải thiện hiệu suất của họ trong trận đấu tiếp theo không?"""
    }
]

chat_text = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    tokenize=False
)

inputs = tokenizer(
    chat_text,
    return_tensors="pt"
).to("cuda:0")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        generation_config=generation_config,
    )
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "assistant\n" in output_text:
    answer = output_text.split("assistant\n")[-1].strip()
else:
    answer = output_text.strip()

print("Assistant reply:", answer)